### Embedding Generation

This notebook loads the preprocessed dataset from Notebook 1 and generates sentence embeddings for all 30 model-configuration combinations. Five multilingual models are evaluated across six text configurations each. 

All embeddings are L2 normalized before saving to ensure cosine similarity calculations in Notebook 3 are numerically consistent.

### Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
import logging
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel, logging as hf_logging
import torch

# Suppress irrelevant warnings
warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'

# Output directory for embeddings
embeddings_dir = r"C:\Users\madee\OneDrive\Desktop\thesis\Thesis Project - Final\data\embeddings"
os.makedirs(embeddings_dir, exist_ok=True)

print("Libraries loaded successfully")
print(f"Embeddings will be saved to: {embeddings_dir}")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries loaded successfully
Embeddings will be saved to: C:\Users\madee\OneDrive\Desktop\thesis\Thesis Project - Final\data\embeddings
CUDA available: True


### Loading the Preprocessed Dataset

The preprocessed dataset produced by Notebook 1 is loaded here. All twelve configuration columns are available and ready for embedding generation.

In [2]:
data_path = r"C:\Users\madee\OneDrive\Desktop\thesis\Thesis Project - Final\data\processed\dataset_preprocessed.csv"
df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape}")
print(f"\nLabel distribution:\n{df['similarity_label'].value_counts()}")
print(f"\nConfiguration columns available:")
config_cols = [c for c in df.columns if c.startswith('config_')]
for col in config_cols:
    print(f"  {col}")

Dataset loaded: (154, 33)

Label distribution:
similarity_label
1    77
0    77
Name: count, dtype: int64

Configuration columns available:
  config_1_fi
  config_1_en
  config_2_fi
  config_2_en
  config_3_fi
  config_3_en
  config_4_fi
  config_4_en
  config_5_fi
  config_5_en
  config_6_fi
  config_6_en


### Defining the Five Models

Five models are evaluated in this study. Four are multilingual sentence embedding models selected to represent different architectural and training strategies. 

The fifth is FinBERT, a Finnish monolingual model included as a baseline to assess whether a language-specific model can compete with multilingual alternatives on Finnish-English cross-lingual similarity.

| Model key | Model name                                      | Type          |
|-----------|-------------------------------------------------|---------------|
| mpnet     | paraphrase-multilingual-mpnet-base-v2           | Multilingual  |
| stsb      | stsb-xlm-r-multilingual                         | Multilingual  |
| e5        | intfloat/multilingual-e5-base                   | Multilingual  |
| labse     | sentence-transformers/LaBSE                     | Multilingual  |
| finbert   | TurkuNLP/bert-base-finnish-cased-v1             | Monolingual   |

In [3]:
models = {
    'mpnet'  : 'paraphrase-multilingual-mpnet-base-v2',
    'stsb'   : 'stsb-xlm-r-multilingual',
    'e5'     : 'intfloat/multilingual-e5-base',
    'labse'  : 'sentence-transformers/LaBSE',
    'finbert': 'TurkuNLP/bert-base-finnish-cased-v1'
}

print("Models defined:")
for key, name in models.items():
    print(f"  {key}: {name}")

Models defined:
  mpnet: paraphrase-multilingual-mpnet-base-v2
  stsb: stsb-xlm-r-multilingual
  e5: intfloat/multilingual-e5-base
  labse: sentence-transformers/LaBSE
  finbert: TurkuNLP/bert-base-finnish-cased-v1


### Embedding Functions

Each model requires a slightly different approach to generate sentence embeddings. 

The four multilingual models are loaded via the entence-transformers library and encode text directly. FinBERT is a standard BERT model without a pooling head, so mean pooling is applied manually over the token embeddings to produce a single 
sentence vector. 

The e5 model requires a "passage: " prefix on all input texts as specified in its original training design.

In [4]:
def encode_sentence_transformer(model, texts):
    return model.encode(texts, batch_size=32, show_progress_bar=False,
                        convert_to_numpy=True)

def encode_finbert(tokenizer, model, texts, device):
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(
                text, return_tensors='pt',
                truncation=True, max_length=512,
                padding=True
            ).to(device)
            outputs = model(**inputs)
            # Mean pooling over token embeddings
            token_embeddings = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']
            mask_expanded = attention_mask.unsqueeze(-1).expand(
                token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            embedding = (sum_embeddings / sum_mask).squeeze().cpu().numpy()
            all_embeddings.append(embedding)
    return np.array(all_embeddings)

def l2_normalize(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / np.clip(norms, a_min=1e-10, a_max=None)

print("Embedding functions defined.")
print("  encode_sentence_transformer: for mpnet, stsb, e5, labse")
print("  encode_finbert: for FinBERT with manual mean pooling")
print("  l2_normalize: applied to all embeddings before saving")

Embedding functions defined.
  encode_sentence_transformer: for mpnet, stsb, e5, labse
  encode_finbert: for FinBERT with manual mean pooling
  l2_normalize: applied to all embeddings before saving


### Generating and Saving Embeddings

Embeddings are generated for all 30 model-configuration combinations. 

For each combination, Finnish and English texts are encoded separately and L2 normalized before saving as NumPy files. 

This produces 60 files in total, two per combination, which serve as the sole input to Notebook 3.

In [5]:
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

configs = [1, 2, 3, 4, 5, 6]

for model_key, model_name in models.items():
    print(f"Loading {model_key}: {model_name}")
    
    if model_key == 'finbert':
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model_obj = AutoModel.from_pretrained(model_name).to(device)
    else:
        model_obj = SentenceTransformer(model_name, device=str(device))
    
    for config_num in configs:
        fi_col = f'config_{config_num}_fi'
        en_col = f'config_{config_num}_en'
        
        fi_texts = df[fi_col].tolist()
        en_texts = df[en_col].tolist()
        
        # Add e5 prefix
        if model_key == 'e5':
            fi_texts = ['passage: ' + t for t in fi_texts]
            en_texts = ['passage: ' + t for t in en_texts]
        
        if model_key == 'finbert':
            fi_emb = encode_finbert(tokenizer, model_obj, fi_texts, device)
            en_emb = encode_finbert(tokenizer, model_obj, en_texts, device)
        else:
            fi_emb = encode_sentence_transformer(model_obj, fi_texts)
            en_emb = encode_sentence_transformer(model_obj, en_texts)
        
        # L2 normalize
        fi_emb = l2_normalize(fi_emb)
        en_emb = l2_normalize(en_emb)
        
        # Save
        fi_path = os.path.join(embeddings_dir, f'{model_key}_config{config_num}_fi.npy')
        en_path = os.path.join(embeddings_dir, f'{model_key}_config{config_num}_en.npy')
        np.save(fi_path, fi_emb)
        np.save(en_path, en_emb)
        
        print(f"  Config {config_num}: saved {fi_emb.shape} Finnish and {en_emb.shape} English")
    
    print(f"  {model_key} complete.\n")
    
    # Free memory
    del model_obj
    torch.cuda.empty_cache()

print("All 60 embedding files saved.")

Using device: cuda

Loading mpnet: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Config 1: saved (154, 768) Finnish and (154, 768) English
  Config 2: saved (154, 768) Finnish and (154, 768) English
  Config 3: saved (154, 768) Finnish and (154, 768) English
  Config 4: saved (154, 768) Finnish and (154, 768) English
  Config 5: saved (154, 768) Finnish and (154, 768) English
  Config 6: saved (154, 768) Finnish and (154, 768) English
  mpnet complete.

Loading stsb: stsb-xlm-r-multilingual


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Config 1: saved (154, 768) Finnish and (154, 768) English
  Config 2: saved (154, 768) Finnish and (154, 768) English
  Config 3: saved (154, 768) Finnish and (154, 768) English
  Config 4: saved (154, 768) Finnish and (154, 768) English
  Config 5: saved (154, 768) Finnish and (154, 768) English
  Config 6: saved (154, 768) Finnish and (154, 768) English
  stsb complete.

Loading e5: intfloat/multilingual-e5-base


...

  Config 1: saved (154, 768) Finnish and (154, 768) English
  Config 2: saved (154, 768) Finnish and (154, 768) English
  Config 3: saved (154, 768) Finnish and (154, 768) English
  Config 4: saved (154, 768) Finnish and (154, 768) English
  Config 5: saved (154, 768) Finnish and (154, 768) English
  Config 6: saved (154, 768) Finnish and (154, 768) English
  e5 complete.

Loading labse: sentence-transformers/LaBSE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Config 1: saved (154, 768) Finnish and (154, 768) English
  Config 2: saved (154, 768) Finnish and (154, 768) English
  Config 3: saved (154, 768) Finnish and (154, 768) English
  Config 4: saved (154, 768) Finnish and (154, 768) English
  Config 5: saved (154, 768) Finnish and (154, 768) English
  Config 6: saved (154, 768) Finnish and (154, 768) English
  labse complete.

Loading finbert: TurkuNLP/bert-base-finnish-cased-v1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Config 1: saved (154, 768) Finnish and (154, 768) English
  Config 2: saved (154, 768) Finnish and (154, 768) English
  Config 3: saved (154, 768) Finnish and (154, 768) English
  Config 4: saved (154, 768) Finnish and (154, 768) English
  Config 5: saved (154, 768) Finnish and (154, 768) English
  Config 6: saved (154, 768) Finnish and (154, 768) English
  finbert complete.

All 60 embedding files saved.


### Verifying All 60 Embedding Files

Before closing this notebook, all 60 saved files are verified to confirm they exist and carry the correct shape of 154 rows and 768 dimensions. Any missing or malformed file would cause silent errors in Notebook 3.

In [6]:
model_keys = ['mpnet', 'stsb', 'e5', 'labse', 'finbert']
configs = [1, 2, 3, 4, 5, 6]
languages = ['fi', 'en']

all_passed = True
print("Verifying all 60 embedding files:\n")

for model_key in model_keys:
    for config_num in configs:
        for lang in languages:
            fname = f'{model_key}_config{config_num}_{lang}.npy'
            fpath = os.path.join(embeddings_dir, fname)
            if not os.path.exists(fpath):
                print(f"  MISSING: {fname}")
                all_passed = False
            else:
                emb = np.load(fpath)
                if emb.shape != (154, 768):
                    print(f"  WRONG SHAPE: {fname} -> {emb.shape}")
                    all_passed = False

if all_passed:
    print("All 60 files verified: shape (154, 768) confirmed for every file.")
else:
    print("Issues found. Fix before proceeding to Notebook 3.")

Verifying all 60 embedding files:

All 60 files verified: shape (154, 768) confirmed for every file.


### Notebook Summary

This notebook generated sentence embeddings for all 30 model-configuration combinations across five models and six text configurations. 

Finnish and English texts were encoded separately, L2 normalized, and saved as NumPy files. FinBERT required manual mean pooling over token embeddings due to its standard BERT architecture. 

The e5 model received a "passage: " prefix on all inputs as required by its training design. 

All 60 files were verified at shape (154, 768) and are ready for threshold testing and evaluation in Notebook 3.